# Data agent: Yahoo symbols and daily OHLC prices

This notebook builds the price data of the 50 Belfius ETFs in two steps.

| Step | Question | Output |
|---|---|---|
| 1. Symbols | Which Yahoo symbol gives the Belfius price of each ETF? | `data/etf_symbols.csv` |
| 2. Prices | What were the daily open, high, low and close of each ETF, in EUR, on the Belfius exchange? Yahoo for 49 ETFs, EODHD for NASDAQ 3X | `data/cache/prices_ohlc_<date>.csv` |

**Rules we follow (official Belfius Q&A)**
- Each ETF is priced on **one reference exchange**: Xetra, except BEL 20 (Euronext Brussels) and COPPER LVG / PALLADIUM LVG (Euronext Milan).
- All those prices are **in EUR**.
- **Real prices**, not adjusted for dividends: Belfius values the portfolio at real closes and does not credit dividends.

**Our own rules: real prices only.**
- We never add, fill or move a price. A day without a quote is simply absent (NA), and listed in the checks.
- An ETF whose exact price we cannot get is **left out and reported**, never replaced by the price of another exchange.
- The technical score is computed ETF by ETF, on each ETF's own last available prices: ETFs do not need to share the same days.

Run the cells one at a time with **Shift + Enter**.

## 0. Setup

In [32]:
import os
import sys
import time
from datetime import date, timedelta

import pandas as pd
import requests
import yfinance as yf
from dotenv import load_dotenv

# The project folder is the one above notebooks/
sys.path.insert(0, os.path.abspath(".."))
from data.config import DATA_DIR
from data.universe import load_universe, reference_exchange

universe = load_universe()
print(len(universe), "ETFs in the Belfius universe")
universe.head()

50 ETFs in the Belfius universe


,id,theme,ticker,isin,name
0,1,Global equity (benchmark),WORLD,IE00B4L5Y983,iShares Core MSCI World UCITS ETF USD (Acc)
1,2,US equity,S&P 500,IE00B5BMR087,iShares Core S&P 500 UCITS ETF USD (Acc)
2,3,European equity,EUROPE 600,LU0908500753,Amundi Core Stoxx Europe 600 UCITS ETF Acc
3,4,European equity,EURO STOXX 50,IE00B53L3W79,iShares Core EURO STOXX 50 UCITS ETF EUR (Acc)
4,5,European equity,IBEX 35,FR0010655746,Amundi IBEX 35 UCITS ETF Acc


## 1. Find the Yahoo symbol of each ETF

**The problem.** Belfius identifies each ETF by its **ISIN** (`IE00B4L5Y983`). Yahoo Finance needs a **symbol** (`EUNL.DE`),
and one ETF has one symbol per exchange where it is listed. We need the symbol of the **Belfius reference exchange**.

**The method, for each ETF:**
1. **OpenFIGI** (free public API from Bloomberg): we send *"this ISIN, on this exchange"* and it returns the ticker of that listing.
2. We add the Yahoo suffix of the exchange (`.DE` Xetra, `.BR` Brussels, `.MI` Milan).
3. We **check on Yahoo** that the symbol has prices, and that they are in EUR.

Nothing is typed by hand: the same code works for any list of ISINs.

**Status of each ETF**

| Status | Meaning | What happens next |
|---|---|---|
| `OK` | Symbol found, prices on Yahoo, in EUR | Used in step 2 |
| `NO DATA` | Symbol found, but Yahoo has no prices for it | Left out, reported |
| `NOT FOUND` | OpenFIGI knows no listing on the reference exchange | Left out, reported |
| `NOT EUR` | Prices found, but not in EUR | Left out, reported |
| `EODHD` | Skipped here on purpose: no Xetra listing on Yahoo | Prices downloaded from EODHD in step 2 |

**Known limits (3 ETFs out of 50)**
- **STOXX SHORT** and **EURO STOXX 2X**: OpenFIGI returns the *Bloomberg* ticker (`BXXP`, `LYXLVE`), which for these two old Lyxor funds
  differs from the Xetra trading symbol that Yahoo uses (`LSK8`, `LYMZ`). **Fixed** in `SYMBOL_FIXES` (cell below),
  with the source of each symbol. The fixed symbols are still checked on Yahoo at every run, like the others.
- **NASDAQ 3X**: Yahoo does not carry its Xetra listing at all. Step 1 **skips it** (`EODHD_SYMBOLS`, cell below):
  no OpenFIGI call, no Yahoo check. Its prices come from **EODHD** in step 2.

In [33]:
# One line per Belfius reference exchange: OpenFIGI exchange code and Yahoo suffix
EXCHANGES = {
    "Xetra":             {"openfigi_code": "GR", "yahoo_suffix": ".DE"},
    "Euronext Brussels": {"openfigi_code": "BB", "yahoo_suffix": ".BR"},
    "Euronext Milan":    {"openfigi_code": "IM", "yahoo_suffix": ".MI"},
}

# Manual fixes, applied instead of OpenFIGI: {Belfius ticker: Yahoo symbol}.
# OpenFIGI returns the Bloomberg ticker; for these two old Lyxor funds it differs
# from the Xetra trading symbol that Yahoo uses.
SYMBOL_FIXES = {
    "STOXX SHORT": "LSK8.DE",     # Bloomberg BXXP GY   - source: live.deutsche-boerse.com, ISIN FR0010424143
    "EURO STOXX 2X": "LYMZ.DE",   # Bloomberg LYXLVE GY - source: Amundi factsheet, ISIN FR0010468983
}

# ETFs whose Xetra listing is not on Yahoo: {Belfius ticker: EODHD symbol}.
# Step 1 skips them (no Yahoo symbol). Step 2 downloads their prices from EODHD instead of Yahoo.
# EODHD free plan, official Deutsche Boerse data distributor: closes checked identical to Xetra
# (11 days, 27/10 to 10/11/2026, gap 0.00 %).
EODHD_SYMBOLS = {
    "NASDAQ 3X": "3QQQ.XETRA",
}

OPENFIGI_URL = "https://api.openfigi.com/v3/mapping"
OPENFIGI_PAUSE_SECONDS = 3   # without an API key, OpenFIGI allows about 25 requests per minute
SYMBOLS_CSV = DATA_DIR / "etf_symbols.csv"

### 1.1 `find_yahoo_symbol()`: ISIN + exchange → Yahoo symbol

If the network drops or OpenFIGI says we asked too often, the function waits and tries again (3 attempts),
instead of stopping the whole loop.

In [34]:
def find_yahoo_symbol(isin, exchange):
    # ISIN + exchange name -> Yahoo symbol, e.g. ('IE00B4L5Y983', 'Xetra') -> 'EUNL.DE'.
    # Returns None when OpenFIGI knows no listing of this ISIN on this exchange.
    openfigi_code = EXCHANGES[exchange]["openfigi_code"]
    yahoo_suffix = EXCHANGES[exchange]["yahoo_suffix"]
    query = [{"idType": "ID_ISIN", "idValue": isin, "exchCode": openfigi_code}]

    for attempt in range(1, 4):
        try:
            response = requests.post(OPENFIGI_URL, json=query, timeout=20)
        except requests.exceptions.ConnectionError:
            print(f"  network error for {isin}, attempt {attempt}/3, waiting 10 s")
            time.sleep(10)
            continue

        if response.status_code == 429:
            print(f"  OpenFIGI rate limit, attempt {attempt}/3, waiting 60 s")
            time.sleep(60)
            continue

        if response.status_code != 200:
            raise RuntimeError(f"OpenFIGI error {response.status_code} for {isin}: {response.text[:200]}")

        answer = response.json()[0]
        if "data" not in answer:
            return None
        ticker = answer["data"][0]["ticker"]
        return ticker + yahoo_suffix

    raise RuntimeError(f"OpenFIGI did not answer for {isin} after 3 attempts")


# Quick test on one ETF
print(find_yahoo_symbol("IE00B4L5Y983", "Xetra"))   # WORLD, expected EUNL.DE

EUNL.DE


### 1.2 `check_on_yahoo()`: does Yahoo have prices for this symbol?

Yahoo sometimes answers empty by mistake (CHINA A failed once, then worked). So when a symbol returns no data,
we wait 5 seconds and ask a second time before concluding.

In [35]:
def check_on_yahoo(symbol):
    # Number of trading days over one year, currency and last date of a Yahoo symbol.
    history = pd.DataFrame()
    for attempt in range(1, 3):
        try:
            history = yf.Ticker(symbol).history(period="1y", auto_adjust=False)
        except Exception:
            history = pd.DataFrame()
        if len(history) > 0:
            break
        time.sleep(5)

    if len(history) == 0:
        return {"days_1y": 0, "currency": None, "last_date": None}

    currency = None
    try:
        currency = yf.Ticker(symbol).fast_info["currency"]
    except Exception:
        currency = None

    last_date = history.index[-1].date()
    return {"days_1y": len(history), "currency": currency, "last_date": last_date}


# Quick test
print(check_on_yahoo("EUNL.DE"))

{'days_1y': 253, 'currency': 'EUR', 'last_date': datetime.date(2026, 11, 11)}


### 1.3 The 50 ETFs

About 3 minutes (one OpenFIGI call every 3 seconds). The result is saved, so you only run this when the ETF list changes.

In [36]:
rows = []

for etf in universe.itertuples():
    exchange = reference_exchange(etf.ticker)

    # 0. ETFs priced by EODHD: nothing to do here, step 2 takes care of them
    if etf.ticker in EODHD_SYMBOLS:
        rows.append({
            "ticker": etf.ticker,
            "isin": etf.isin,
            "exchange": exchange,
            "yahoo_symbol": None,
            "source": "EODHD",
            "status": "EODHD",
            "currency": None,
            "days_1y": 0,
            "last_date": None,
        })
        print(f"{etf.ticker:18} {exchange:18} {'-':11} EODHD     prices from EODHD in step 2")
        continue

    # 1. Yahoo symbol: manual fix if there is one, otherwise OpenFIGI
    if etf.ticker in SYMBOL_FIXES:
        symbol = SYMBOL_FIXES[etf.ticker]
        source = "manual fix"
    else:
        symbol = find_yahoo_symbol(etf.isin, exchange)
        source = "OpenFIGI"
        time.sleep(OPENFIGI_PAUSE_SECONDS)

    # 2. Check on Yahoo
    if symbol is None:
        check = {"days_1y": 0, "currency": None, "last_date": None}
        status = "NOT FOUND"
    else:
        check = check_on_yahoo(symbol)
        if check["days_1y"] == 0:
            status = "NO DATA"
        elif check["currency"] != "EUR":
            status = "NOT EUR"
        else:
            status = "OK"

    rows.append({
        "ticker": etf.ticker,
        "isin": etf.isin,
        "exchange": exchange,
        "yahoo_symbol": symbol,
        "source": source,
        "status": status,
        "currency": check["currency"],
        "days_1y": check["days_1y"],
        "last_date": check["last_date"],
    })
    print(f"{etf.ticker:18} {exchange:18} {str(symbol):11} {status:9} {check['days_1y']:4} days  {check['currency']}")

symbols = pd.DataFrame(rows)
symbols.to_csv(SYMBOLS_CSV, index=False)
print(f"\nSaved to {SYMBOLS_CSV}")

WORLD              Xetra              EUNL.DE     OK         253 days  EUR
S&P 500            Xetra              SXR8.DE     OK         253 days  EUR
EUROPE 600         Xetra              LYP6.DE     OK         253 days  EUR
EURO STOXX 50      Xetra              SXRT.DE     OK         253 days  EUR
IBEX 35            Xetra              AMES.DE     OK         253 days  EUR
BEL 20             Euronext Brussels  BEL.BR      OK         256 days  EUR
WORLD SMALL CAP    Xetra              IUSN.DE     OK         253 days  EUR
WORLD MIN VOL      Xetra              IQQ0.DE     OK         253 days  EUR
DIVIDEND LEADERS   Xetra              VDIV.DE     OK         253 days  EUR
NASDAQ INCOME      Xetra              JEQP.DE     OK         253 days  EUR
NASDAQ             Xetra              SXRV.DE     OK         254 days  EUR
SEMIS              Xetra              SEC0.DE     OK         253 days  EUR
CYBERSEC           Xetra              USPY.DE     OK         253 days  EUR
BIOTECH            Xetra 

To skip the 3 minutes next time, reload the saved table instead:

In [37]:
# symbols = pd.read_csv(SYMBOLS_CSV)

### 1.4 Result

**Expected:** 49 `OK`, and 1 `EODHD` (NASDAQ 3X, priced from EODHD in step 2).

In [38]:
print(symbols["status"].value_counts().to_string())
symbols[symbols["status"] != "OK"]

status
OK       49
EODHD     1


,ticker,isin,exchange,yahoo_symbol,source,status,currency,days_1y,last_date
47,NASDAQ 3X,IE00BLRPRL42,Xetra,NaN,EODHD,EODHD,NaN,0,None


## 2. Download the daily OHLC prices

**What we download.** Daily **open, high, low, close**, real prices, in EUR, from one of two sources:

| ETFs | Source | Function |
|---|---|---|
| status `OK` (49 ETFs) | Yahoo Finance (`auto_adjust=False`) | `download_ohlc()` |
| in `EODHD_SYMBOLS` (NASDAQ 3X) | EODHD, official Xetra prices | `download_ohlc_eodhd()` |

Both functions return the same table (columns `Open`, `High`, `Low`, `Close`, one row per day),
so everything after the download is identical for all ETFs.

**Period.** From 2020 to **yesterday**: today's row would be an intraday price while the market is open, not a close.
From 2020, to have enough history for the 200-day average, the 12-month momentum and a backtest over several years.
Recent ETFs (SPACE, QUANTUM…) simply start later. NASDAQ 3X starts one year ago: the free EODHD plan only gives the past year
(enough: its technical indicators are computed on the Nasdaq 100, and its own price is used for the December decisions and valuation).

**Why OHLC and not only the close.** The close is what Belfius uses to execute and value orders.
Open, high and low are useful for the technical agent (true range, volatility, intraday moves) and for the Risk Manager (stops).

In [39]:
START = date(2020, 1, 1)
END = date.today() - timedelta(days=1)   # closed days only: today's row would be an intraday price, not a close


def download_ohlc(symbol, start, end):
    # Daily open, high, low and close of one Yahoo symbol, real prices. `end` is included.
    raw = yf.download(symbol, start=start, end=end + timedelta(days=1),
                      auto_adjust=False, progress=False)
    if raw.empty:
        return raw

    # Recent yfinance versions add the symbol as a second column level: keep only the field names
    if isinstance(raw.columns, pd.MultiIndex):
        raw.columns = raw.columns.get_level_values(0)

    ohlc = raw[["Open", "High", "Low", "Close"]].copy()
    ohlc.index = pd.to_datetime(ohlc.index).tz_localize(None).normalize()
    ohlc = ohlc.dropna(subset=["Close"])
    return ohlc


# Quick test
download_ohlc("EUNL.DE", date(2026, 11, 2), date(2026, 11, 6))

Price,Open,High,Low,Close
Date,,,,
2026-11-02,129.505005,130.294998,129.139999,129.929993
2026-11-03,128.330002,129.300003,128.304993,128.975006
2026-11-04,128.505005,130.735001,128.505005,130.479996
2026-11-05,130.429993,131.035004,126.300003,127.790001
2026-11-06,125.769997,127.404999,125.699997,127.315002


### Prices from EODHD (ETFs in `EODHD_SYMBOLS` only)

**Why.** Yahoo has no Xetra listing for NASDAQ 3X, and Belfius values it at its Xetra close.
EODHD is an official distributor of Deutsche Boerse data, with a free plan (20 calls a day, past year of history).
We checked its closes against the official Xetra closes on 11 days: identical.

**The API key** is read from the `.env` file at the project root (`EODHD_API_KEY=...`), never written in the notebook.
Each run of step 2 uses **one** call for NASDAQ 3X.


In [40]:
EODHD_URL = "https://eodhd.com/api/eod/"
EODHD_FREE_HISTORY_DAYS = 365   # the free plan only gives the past year

# Read the API key from the .env file at the project root (never printed)
load_dotenv(os.path.join("..", ".env"), override=True)
EODHD_API_KEY = os.getenv("EODHD_API_KEY")
if EODHD_API_KEY is None or EODHD_API_KEY == "":
    print("EODHD_API_KEY is missing from .env: the EODHD ETFs will be left out")
else:
    print("EODHD API key found")


def download_ohlc_eodhd(eodhd_symbol, start, end):
    # Daily open, high, low and close of one EODHD symbol (official Xetra prices). `end` is included.
    # Same output as download_ohlc(): columns Open, High, Low, Close, one row per day.
    if EODHD_API_KEY is None or EODHD_API_KEY == "":
        return pd.DataFrame()

    # The free plan only covers the past year
    earliest = end - timedelta(days=EODHD_FREE_HISTORY_DAYS)
    if start < earliest:
        start = earliest

    params = {
        "api_token": EODHD_API_KEY,
        "fmt": "json",
        "from": start.isoformat(),
        "to": end.isoformat(),
    }
    response = requests.get(EODHD_URL + eodhd_symbol, params=params, timeout=30)
    if response.status_code != 200:
        # Do not print the request URL: it contains the API key
        print(f"  EODHD error {response.status_code} for {eodhd_symbol}")
        return pd.DataFrame()

    rows = []
    for bar in response.json():
        rows.append({
            "date": bar["date"],
            "Open": bar["open"],
            "High": bar["high"],
            "Low": bar["low"],
            "Close": bar["close"],
        })
    if len(rows) == 0:
        return pd.DataFrame()

    ohlc = pd.DataFrame(rows)
    ohlc.index = pd.to_datetime(ohlc["date"])
    ohlc = ohlc[["Open", "High", "Low", "Close"]]
    ohlc = ohlc.dropna(subset=["Close"])
    return ohlc


EODHD API key found


In [41]:
# ETFs with a price source: Yahoo (status OK) or EODHD (status EODHD)
ok_etfs = symbols[symbols["status"].isin(["OK", "EODHD"])]

downloaded = {}   # Belfius ticker -> OHLC table

for etf in ok_etfs.itertuples():
    if etf.ticker in EODHD_SYMBOLS:
        source = "EODHD"
        source_symbol = EODHD_SYMBOLS[etf.ticker]
        ohlc = download_ohlc_eodhd(source_symbol, START, END)
    else:
        source = "Yahoo"
        source_symbol = etf.yahoo_symbol
        ohlc = download_ohlc(source_symbol, START, END)

    if ohlc.empty:
        print(f"{etf.ticker:18} {source:6} {source_symbol:11} no data for this period")
        continue
    downloaded[etf.ticker] = ohlc
    print(f"{etf.ticker:18} {source:6} {source_symbol:11} {len(ohlc):5} days  "
          f"from {ohlc.index[0].date()} to {ohlc.index[-1].date()}")

print(f"\n{len(downloaded)} ETFs downloaded")


WORLD              Yahoo  EUNL.DE      1746 days  from 2020-01-02 to 2026-11-10
S&P 500            Yahoo  SXR8.DE      1746 days  from 2020-01-02 to 2026-11-10
EUROPE 600         Yahoo  LYP6.DE       691 days  from 2024-02-19 to 2026-11-10
EURO STOXX 50      Yahoo  SXRT.DE      1746 days  from 2020-01-02 to 2026-11-10
IBEX 35            Yahoo  AMES.DE       652 days  from 2024-04-16 to 2026-11-10
BEL 20             Yahoo  BEL.BR       1758 days  from 2020-01-02 to 2026-11-10
WORLD SMALL CAP    Yahoo  IUSN.DE      1746 days  from 2020-01-02 to 2026-11-10
WORLD MIN VOL      Yahoo  IQQ0.DE      1747 days  from 2020-01-02 to 2026-11-10
DIVIDEND LEADERS   Yahoo  VDIV.DE      1746 days  from 2020-01-02 to 2026-11-10
NASDAQ INCOME      Yahoo  JEQP.DE       509 days  from 2024-11-04 to 2026-11-10
NASDAQ             Yahoo  SXRV.DE      1748 days  from 2020-01-02 to 2026-11-10
SEMIS              Yahoo  SEC0.DE      1341 days  from 2021-08-06 to 2026-11-10
CYBERSEC           Yahoo  USPY.DE      1

### 2.1 One table of real prices

Each ETF keeps **its own trading days on its reference exchange**: Xetra for most ETFs, Euronext Brussels for BEL 20,
Euronext Milan for COPPER LVG and PALLADIUM LVG. These are the days Belfius can execute and value each ETF.

**Nothing is added, filled or removed.** If a source has no quote for a day, the row is simply absent:
in the wide view (one column per ETF) it shows as NA. The checks in step 3 list those days.

Known case: Yahoo has no quote on **06/03/2026** for 43 of the 50 ETFs (checked by hand on the Yahoo website,
while Deutsche Boerse publishes a Xetra close that day). The day is left absent for those ETFs.

In [42]:
frames = []

for ticker in downloaded:
    ohlc = downloaded[ticker]
    frames.append(pd.DataFrame({
        "date": ohlc.index.date,
        "ticker": ticker,
        "open_eur": ohlc["Open"].round(4).values,
        "high_eur": ohlc["High"].round(4).values,
        "low_eur": ohlc["Low"].round(4).values,
        "close_eur": ohlc["Close"].round(4).values,
    }))

prices = pd.concat(frames, ignore_index=True)
prices = prices.sort_values(["ticker", "date"]).reset_index(drop=True)
print(len(prices), "rows,", prices["ticker"].nunique(), "ETFs")
prices.tail()

73273 rows, 50 ETFs


,date,ticker,open_eur,high_eur,low_eur,close_eur
73268,2026-11-04,WORLD SMALL CAP,9.141,9.248,9.136,9.208
73269,2026-11-05,WORLD SMALL CAP,9.251,9.300,9.015,9.073
73270,2026-11-06,WORLD SMALL CAP,8.903,9.020,8.893,9.017
73271,2026-11-09,WORLD SMALL CAP,9.067,9.189,9.046,9.183
73272,2026-11-10,WORLD SMALL CAP,9.236,9.250,9.149,9.170


## 3. Checks

The checks **report**, they never change the data.

1. **Coverage:** first and last date, number of days per ETF.
2. **Freshness:** every ETF should have a price on the most recent day. The key check before each trading day.
3. **Missing days:** over the last 12 months, days where some ETFs have a price and others do not.
4. **Consistency of each bar:** prices > 0, high ≥ open and close, low ≤ open and close.
5. **Big moves:** daily moves above 20 %, to check they are real market moves.
6. **ETFs left out**, with the reason.

In [43]:
# 1. Coverage per ETF
coverage = prices.groupby("ticker").agg(
    first=("date", "min"),
    last=("date", "max"),
    days=("date", "size"),
    last_close=("close_eur", "last"),
)
coverage.sort_values("days")

,first,last,days,last_close
ticker,,,,
NASDAQ 3X,2025-11-11,2026-11-10,253,292.4600
QUANTUM,2025-05-26,2026-11-10,372,22.2050
BITCOIN,2025-03-24,2026-11-10,414,6.5170
OVERNIGHT CASH,2025-03-17,2026-11-10,421,110.4630
NASDAQ INCOME,2024-11-04,2026-11-10,509,22.9700
IBEX 35,2024-04-16,2026-11-10,652,518.7000
EUROPE 600,2024-02-19,2026-11-10,691,325.0500
DEFENSE,2023-03-31,2026-11-10,916,50.9300
URANIUM,2023-02-03,2026-11-10,956,37.1750


In [44]:
# 2. Freshness: ETFs without a price on the most recent day of the table
latest_day = prices["date"].max()

stale = []
for ticker in downloaded:
    last_day = prices[prices["ticker"] == ticker]["date"].max()
    if last_day < latest_day:
        stale.append({"ticker": ticker, "last_day": last_day})

print("Most recent day in the table:", latest_day)
print(len(stale), "ETFs without a price on that day (a holiday on their own exchange, or a source problem)")
pd.DataFrame(stale)

Most recent day in the table: 2026-11-10
0 ETFs without a price on that day (a holiday on their own exchange, or a source problem)


""


In [45]:
# 3. Missing days over the last 12 months
one_year_ago = END - timedelta(days=365)
recent = prices[prices["date"] >= one_year_ago]
recent_tickers = sorted(recent["ticker"].unique())

for day, group in recent.groupby("date"):
    present = set(group["ticker"])
    missing = []
    for ticker in recent_tickers:
        if ticker not in present:
            missing.append(ticker)
    if len(missing) == 0:
        continue
    if len(missing) <= 5:
        print(f"{day}  {len(missing):2} ETFs without a price: {missing}")
    else:
        print(f"{day}  {len(missing):2} ETFs without a price (only {len(present)} quoted: {sorted(present)})")

2025-11-10   1 ETFs without a price: ['NASDAQ 3X']
2025-12-24  49 ETFs without a price (only 1 quoted: ['BEL 20'])
2025-12-31  49 ETFs without a price (only 1 quoted: ['BEL 20'])
2026-03-06  43 ETFs without a price (only 7 quoted: ['BEL 20', 'EUR GOVT BOND', 'GLOBAL AGG BOND', 'HIGHY BOND', 'NASDAQ', 'NASDAQ 3X', 'OVERNIGHT CASH'])


In [46]:
# 4. Consistency of each bar
not_positive = (prices[["open_eur", "high_eur", "low_eur", "close_eur"]] <= 0).any(axis=1)
high_too_low = prices["high_eur"] < prices[["open_eur", "close_eur"]].max(axis=1)
low_too_high = prices["low_eur"] > prices[["open_eur", "close_eur"]].min(axis=1)

print("Bars with a price <= 0:      ", int(not_positive.sum()))
print("Bars with high < open/close: ", int(high_too_low.sum()))
print("Bars with low > open/close:  ", int(low_too_high.sum()))

problems = prices[not_positive | high_too_low | low_too_high]
problems.head(20)

Bars with a price <= 0:       0
Bars with high < open/close:  155
Bars with low > open/close:   165


,date,ticker,open_eur,high_eur,low_eur,close_eur
864,2023-05-15,BEL 20,55.38,55.48,55.38,55.52
868,2023-05-19,BEL 20,55.17,55.39,54.97,55.45
871,2023-05-24,BEL 20,54.71,54.71,54.20,54.16
872,2023-05-25,BEL 20,54.32,54.32,53.92,53.78
873,2023-05-26,BEL 20,53.68,54.13,53.41,54.17
875,2023-05-30,BEL 20,54.01,54.01,53.17,53.14
878,2023-06-02,BEL 20,53.13,53.88,53.13,53.96
884,2023-06-12,BEL 20,54.22,54.30,54.22,54.19
887,2023-06-15,BEL 20,54.32,54.32,54.10,54.38
899,2023-07-03,BEL 20,53.91,53.91,52.68,52.65


In [47]:
# 5. Big daily moves, computed on each ETF's own series (previous available close)
BIG_MOVE_PERCENT = 20

big_moves = []
for ticker in downloaded:
    ticker_rows = prices[prices["ticker"] == ticker]
    previous_close = None
    for row in ticker_rows.itertuples():
        if previous_close is not None:
            move = (row.close_eur / previous_close - 1) * 100
            if abs(move) > BIG_MOVE_PERCENT:
                big_moves.append({"ticker": ticker, "date": row.date,
                                  "previous_close": previous_close, "close": row.close_eur,
                                  "move_percent": round(move, 2)})
        previous_close = row.close_eur

print(len(big_moves), "daily moves above", BIG_MOVE_PERCENT, "%")
pd.DataFrame(big_moves)

26 daily moves above 20 %


,ticker,date,previous_close,close,move_percent
0,EURO BANKS,2026-10-26,349.8000,4.4275,-98.73
1,BRAZIL,2020-03-12,21.0700,16.5100,-21.64
2,OIL & GAS E&P,2020-03-09,9.5450,6.9630,-27.05
3,HYDROGEN,2020-03-24,133.0600,160.0000,20.25
4,STOXX SHORT,2020-03-12,4.6215,5.7390,24.18
5,COPPER LVG,2022-11-04,11.7240,14.2120,21.22
6,COPPER LVG,2025-04-04,17.6680,13.7120,-22.39
7,COPPER LVG,2025-07-09,15.7080,19.5500,24.46
8,COPPER LVG,2025-07-31,21.1350,7.5060,-64.49
9,PALLADIUM LVG,2020-03-12,545.0300,362.2800,-33.53


In [48]:
# 6. ETFs left out
left_out = []
for etf in symbols.itertuples():
    if etf.ticker not in downloaded:
        left_out.append({"ticker": etf.ticker, "yahoo_symbol": etf.yahoo_symbol, "status": etf.status})

left_out = pd.DataFrame(left_out)
print(len(downloaded), "/", len(symbols), "ETFs priced")
left_out

50 / 50 ETFs priced


""


## 4. Look at the data

The views below are the ones to show when presenting the table.

**The format: one row per day and per ETF** ("long" format), with the Belfius name in `ticker`.
This is the format shared by the whole team (`date`, `ticker`, ...): easy to filter by ETF or by day,
and adding an ETF just adds rows, never columns.

In [49]:
df = prices.set_index(["date", "ticker"]).sort_index()
df.head(10)

open_eur  high_eur  low_eur  close_eur
date       ticker                                                  
2020-01-02 BEL 20              58.210    59.480   58.210     59.430
           BIOTECH             34.680    34.830   34.025     34.025
           BRAZIL              32.335    32.740   32.335     32.580
           CHINA A              3.992     4.030    3.992      4.021
           CHINA LC           112.940   114.120  112.680    113.840
           COPPER LVG          11.248    11.320   10.966     11.084
           CYBERSEC            15.020    15.190   15.004     15.076
           DIVIDEND LEADERS    29.535    29.850   29.535     29.725
           EMMKT               27.181    27.481   27.154     27.372
           EUR GOVT BOND      130.330   130.360  129.735    130.300

In [50]:
# The first rows
prices.head(10)

,date,ticker,open_eur,high_eur,low_eur,close_eur
0,2020-01-02,BEL 20,58.21,59.48,58.21,59.43
1,2020-01-03,BEL 20,59.12,59.14,58.95,59.08
2,2020-01-06,BEL 20,58.60,58.77,57.94,58.72
3,2020-01-07,BEL 20,58.99,59.30,58.92,58.96
4,2020-01-08,BEL 20,58.65,58.88,58.38,58.88
5,2020-01-09,BEL 20,59.57,59.60,59.30,59.32
6,2020-01-10,BEL 20,59.34,59.42,59.33,59.37
7,2020-01-13,BEL 20,59.67,59.67,58.92,58.92
8,2020-01-14,BEL 20,58.95,59.01,58.82,59.01
9,2020-01-15,BEL 20,58.90,58.99,58.66,58.78


In [51]:
# Size, period and column types
print("Rows: ", len(prices))
print("ETFs: ", prices["ticker"].nunique())
print("From ", prices["date"].min(), "to", prices["date"].max())
prices.dtypes

Rows:  73273
ETFs:  50
From  2020-01-02 to 2026-11-10


date          object
ticker           str
open_eur     float64
high_eur     float64
low_eur      float64
close_eur    float64
dtype: object

### 4.1 One ETF: its last 10 days

This is the history the technical agent reads for one ETF.

In [52]:
one_etf = prices[prices["ticker"] == "WORLD"]
one_etf.tail(10)

,date,ticker,open_eur,high_eur,low_eur,close_eur
68024,2026-10-28,WORLD,131.825,131.900,131.490,131.720
68025,2026-10-29,WORLD,131.470,132.200,131.350,131.460
68026,2026-10-30,WORLD,130.110,130.460,129.470,129.685
68027,2026-11-02,WORLD,129.505,130.295,129.140,129.930
68028,2026-11-03,WORLD,128.330,129.300,128.305,128.975
68029,2026-11-04,WORLD,128.505,130.735,128.505,130.480
68030,2026-11-05,WORLD,130.430,131.035,126.300,127.790
68031,2026-11-06,WORLD,125.770,127.405,125.700,127.315
68032,2026-11-09,WORLD,126.545,128.830,126.525,128.445
68033,2026-11-10,WORLD,128.965,129.100,128.230,128.650


### 4.2 One day: all ETFs

This is what the agents see on a decision day: the last bar of every ETF.
An ETF missing from this list had no quote that day (see the freshness check).
On a trading day T, Belfius executes the order at the close of T-1, so this close is the execution price.

In [53]:
last_day = prices["date"].max()
snapshot = prices[prices["date"] == last_day].set_index("ticker")
print("Day:", last_day, "-", len(snapshot), "ETFs")
snapshot[["open_eur", "high_eur", "low_eur", "close_eur"]]

Day: 2026-11-10 - 50 ETFs


,open_eur,high_eur,low_eur,close_eur
ticker,,,,
BEL 20,92.0100,92.3300,91.7500,91.7500
BIOTECH,69.2900,69.4500,68.4600,68.4800
BITCOIN,6.6350,6.6670,6.4950,6.5170
BRAZIL,23.4500,23.4500,23.0350,23.2050
CHINA A,4.6915,4.7130,4.6845,4.7025
CHINA LC,87.9500,88.2500,87.7600,87.7600
CLEAN ENERGY,13.3720,13.4400,13.1700,13.2140
COPPER LVG,14.3880,14.4480,13.9020,13.9020
COPPER MINERS,52.2800,52.3900,50.8700,51.1300


### 4.3 The closes side by side: one column per ETF

The same data in "wide" format. It is the most convenient shape to compute indicators
(moving averages, momentum, volatility) on all ETFs at once.

A day without a quote for an ETF shows as **NaN** (NA). To compute an indicator on the last available prices
of one ETF, drop its NA first, e.g. `closes["BEL 20"].dropna().rolling(20).mean()`.

In [54]:
closes = prices.pivot(index="date", columns="ticker", values="close_eur")
print(closes.shape[0], "days x", closes.shape[1], "ETFs")
closes.tail()

1758 days x 50 ETFs


ticker,BEL 20,BIOTECH,BITCOIN,BRAZIL,CHINA A,CHINA LC,CLEAN ENERGY,COPPER LVG,COPPER MINERS,CRYPTO EQUITY,...,SILVER MINERS,SPACE,STAPLES,STOXX SHORT,URANIUM,US TREASURY 20Y,WORLD,WORLD ENERGY,WORLD MIN VOL,WORLD SMALL CAP
date,,,,,,,,,,,,,,,,,,,,,
2026-11-04,89.73,68.21,7.1640,22.875,4.7220,87.67,13.338,14.894,51.54,8.937,...,31.895,59.86,54.10,0.4155,36.645,87.896,130.480,68.68,72.20,9.208
2026-11-05,90.43,67.43,6.8225,22.770,4.6910,88.62,12.826,14.414,50.49,8.249,...,32.265,56.62,54.10,0.4189,35.030,88.470,127.790,69.00,71.99,9.073
2026-11-06,89.87,67.46,6.7830,23.135,4.7165,89.52,13.018,14.192,50.11,8.461,...,33.015,58.86,53.39,0.4304,36.125,88.942,127.315,67.88,71.26,9.017
2026-11-09,91.44,68.90,6.6765,23.350,4.7355,88.01,13.290,14.636,52.25,8.834,...,34.740,64.61,53.00,0.4133,37.735,88.474,128.445,67.32,71.40,9.183
2026-11-10,91.75,68.48,6.5170,23.205,4.7025,87.76,13.214,13.902,51.13,8.701,...,32.985,64.53,53.46,0.4083,37.175,87.716,128.650,67.26,71.77,9.170


### 4.4 Daily returns, in %

Each return is computed on the ETF's **own series**: today's close against its previous available close.
The moves should be plausible: a few % for most ETFs, more for the leveraged ones and BITCOIN.

In [55]:
returns_long = prices.copy()
returns_long["previous_close"] = returns_long.groupby("ticker")["close_eur"].shift(1)
returns_long["return_percent"] = (returns_long["close_eur"] / returns_long["previous_close"] - 1) * 100

returns = returns_long.pivot(index="date", columns="ticker", values="return_percent")
returns.tail(5).round(2)

ticker,BEL 20,BIOTECH,BITCOIN,BRAZIL,CHINA A,CHINA LC,CLEAN ENERGY,COPPER LVG,COPPER MINERS,CRYPTO EQUITY,...,SILVER MINERS,SPACE,STAPLES,STOXX SHORT,URANIUM,US TREASURY 20Y,WORLD,WORLD ENERGY,WORLD MIN VOL,WORLD SMALL CAP
date,,,,,,,,,,,,,,,,,,,,,
2026-11-04,2.35,0.32,3.77,0.88,2.25,3.96,0.65,-1.26,2.57,-1.79,...,6.14,-4.16,2.72,-4.28,-2.03,-0.07,1.17,-4.72,0.88,0.32
2026-11-05,0.78,-1.14,-4.77,-0.46,-0.66,1.08,-3.84,-3.22,-2.04,-7.70,...,1.16,-5.41,0.00,0.82,-4.41,0.65,-2.06,0.47,-0.29,-1.47
2026-11-06,-0.62,0.04,-0.58,1.60,0.54,1.02,1.50,-1.54,-0.75,2.57,...,2.32,3.96,-1.31,2.75,3.13,0.53,-0.37,-1.62,-1.01,-0.62
2026-11-09,1.75,2.13,-1.57,0.93,0.40,-1.69,2.09,3.13,4.27,4.41,...,5.22,9.77,-0.73,-3.97,4.46,-0.53,0.89,-0.82,0.20,1.84
2026-11-10,0.34,-0.61,-2.39,-0.62,-0.70,-0.28,-0.57,-5.02,-2.14,-1.51,...,-5.05,-0.12,0.87,-1.21,-1.48,-0.86,0.16,-0.09,0.52,-0.14


In [56]:
# The 10 biggest daily moves over the whole period
with_returns = returns_long.dropna(subset=["return_percent"])
order = with_returns["return_percent"].abs().sort_values(ascending=False).index
biggest = with_returns.loc[order].head(10)
biggest[["date", "ticker", "previous_close", "close_eur", "return_percent"]].round(2)

,date,ticker,previous_close,close_eur,return_percent
24723,2026-10-26,EURO BANKS,349.80,4.43,-98.73
48586,2022-12-19,PALLADIUM LVG,153.98,35.50,-76.95
12094,2025-07-31,COPPER LVG,21.14,7.51,-64.49
47887,2020-03-25,PALLADIUM LVG,334.50,484.38,44.81
47886,2020-03-24,PALLADIUM LVG,247.83,334.50,34.97
47878,2020-03-12,PALLADIUM LVG,545.03,362.28,-33.53
47879,2020-03-13,PALLADIUM LVG,362.28,257.06,-29.04
49352,2025-12-29,PALLADIUM LVG,22.14,15.72,-28.97
48389,2022-03-14,PALLADIUM LVG,498.44,355.11,-28.76
45708,2020-03-09,OIL & GAS E&P,9.54,6.96,-27.05


### Talking points

- **50 / 50 ETFs** priced automatically, from their ISIN, on the Belfius exchange, in EUR, at real prices.
- **OHLC**: the close for orders and valuation; open, high and low for the technical agent and the Risk Manager.
- **Real prices only**: each ETF on its own exchange's trading days, nothing added, filled or removed.
  Missing days are listed, e.g. Yahoo has no quote on 06/03/2026 for 43 ETFs (checked by hand).
- **2 symbols fixed** (STOXX SHORT, EURO STOXX 2X): OpenFIGI gave the Bloomberg ticker, not the Xetra trading symbol.
- **1 ETF from a second source** (NASDAQ 3X): Yahoo has no Xetra price for it, so it comes from EODHD,
  an official Deutsche Boerse distributor (free plan). Closes checked identical to Xetra.

## 5. Save

`data/cache/` is not committed (`.gitignore`): each person regenerates the prices by running this notebook.

In [57]:
output = DATA_DIR / "cache" / f"prices_ohlc_{END.isoformat()}.csv"
output.parent.mkdir(exist_ok=True)
prices.to_csv(output, index=False)
print("Saved to", output)

Saved to C:\dev\qdc-trading\data\cache\prices_ohlc_2026-11-10.csv


## Summary

**What this notebook gives the rest of the team:** one table, one row per day and per ETF,
with `date`, `ticker` (Belfius name), `open_eur`, `high_eur`, `low_eur`, `close_eur`,
in EUR, at real prices, on each ETF's own trading days. Nothing is filled: a missing day is absent (NA).

**Two sources, one table**
- Yahoo for 49 ETFs, EODHD for NASDAQ 3X. Both give the same columns, so all the steps after the download are shared.
- If a source fails one day, the ETF is simply **not in the table** and listed in step 3: the pipeline does not crash
  and does not invent prices. An ETF without a price is **not tradable**: we never buy what we cannot value.

**Next steps**
1. Confirm with the coach that the free EODHD plan is accepted as a public data source.
2. Once this notebook is validated, move the code into `data/data_agent.py` and add the OHLC columns
   to the Prices format in `data/schemas.py` (shared file: pull request with the team).